# Fine-tuning & comparing the best captioning architectures

The architecture search picked **CNN+GPT-2**, **ViT+GPT-2**, and **CLIP+GPT-2** as the strongest transformer combinations. This notebook treats them the same way the earlier CNN+RNN notebook treated the GRU:

1. **Hyperparameter search** — a short randomized grid search over the GPT-2 decoder's knobs (learning rate, weight decay, dropout, and whether to freeze the GPT-2 base and train only cross-attention).
2. **Full training of the winner** — the best config of each architecture is retrained for **15 epochs with early stopping** (monitor val BLEU-4, patience 3), keeping/saving the per-epoch best weights and rolling back to them.
3. **Head-to-head plot** — the earlier **GRU** winner (`ft5/gru/lr1e-3/h512/e256/d0.5/L2`) is retrained with the *same* 15-epoch + early-stop recipe, and all validation BLEU-4 curves are drawn on one graph.

All runs use the same data subset, batch size, and per-epoch batch cap, so the comparison is apples-to-apples. Encoders are frozen feature extractors. Built for Colab (also runs on a local Jupyter).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy, itertools
import random as _random
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Project settings

In [ ]:
# Portable paths: works on Colab AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    BASE_DIR = "/content"
except ImportError:
    BASE_DIR = os.path.abspath(".")

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_finetune")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

COCO_SPLIT = "val"      # "val" for a smaller project run; "train" for full COCO if downloaded
FAST_DEV_RUN = False    # True only while debugging

# ---- Final training budget (same recipe as the GRU notebook) ----------------
EPOCHS = 1 if FAST_DEV_RUN else 15                 # full training length
EARLY_STOP_PATIENCE = 3
EARLY_STOP_MIN_DELTA = 1e-3
FULL_TRAIN_MAX_BATCHES = 3 if FAST_DEV_RUN else None  # None = full train set each epoch (like the GRU final retrain)

# ---- Hyperparameter-search budget (short trials, like the GRU grid search) ---
GRID_EPOCHS = 1 if FAST_DEV_RUN else 2
GRID_MAX_TRAIN_BATCHES = 3 if FAST_DEV_RUN else 150
MAX_TRIALS = 1 if FAST_DEV_RUN else 4              # sampled configs per architecture

# ---- Shared knobs (kept identical across architectures for fairness) ---------
BATCH_SIZE = 32         # matches the earlier GRU grid search (GRID_BATCH_SIZE=32)
EVAL_NUM_BATCHES = 3 if FAST_DEV_RUN else 10       # val batches used for BLEU
MAX_TEXT_LEN = 40
MAX_GEN_LEN = 40
NUM_WORKERS = 2

# GRU baseline config = the earlier grid-search winner
# best_t16_ft5_gru_lr0.001_h512_e256_d0.5_L2
GRU_EMBED_SIZE = 256
GRU_HIDDEN_SIZE = 512
GRU_NUM_LAYERS = 2
GRU_DROPOUT = 0.5
GRU_LR = 1e-3
GRU_FREQ_THRESHOLD = 5

print("Output dir:", OUTPUT_DIR)
print("Final training:", EPOCHS, "epochs | early-stop patience", EARLY_STOP_PATIENCE)
print("Batches/epoch:", "full train set" if FULL_TRAIN_MAX_BATCHES is None else FULL_TRAIN_MAX_BATCHES)
print("Batch size:", BATCH_SIZE)
print("Search:", MAX_TRIALS, "trials x", GRID_EPOCHS, "epochs x", GRID_MAX_TRAIN_BATCHES, "batches")

## 3. Download MS-COCO data

Pure-Python download/extract (`urllib` + `zipfile`), so it runs on Colab or locally. Existence checks skip the work if the data is already present (reused if you ran an earlier notebook in the same session).

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and split by image id

Leakage-safe split by **image id** so an image never appears in both train and validation.

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]
val_annotations = [a for a in coco_data["annotations"] if a["image_id"] in val_img_ids]

assert train_img_ids.isdisjoint(val_img_ids)
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))
print("Train captions:", len(train_annotations), " Val captions:", len(val_annotations))

## 5. Dataset

Returns the raw PIL image, the caption string, and the image id. Encoder-specific image preprocessing and decoder-specific text encoding happen in the per-architecture `collate`.

In [ ]:
class CocoCaptionDataset(Dataset):
    """Returns (PIL image, caption string, image_id)."""
    def __init__(self, img_dir, annotations, img_id_to_filename):
        self.img_dir = img_dir
        self.annotations = list(annotations)
        self.img_id_to_filename = dict(img_id_to_filename)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        img_id = ann["image_id"]
        path = os.path.join(self.img_dir, self.img_id_to_filename[img_id])
        image = Image.open(path).convert("RGB")
        return image, ann["caption"], img_id

train_dataset = CocoCaptionDataset(img_dir, train_annotations, img_id_to_filename)
val_dataset = CocoCaptionDataset(img_dir, val_annotations, img_id_to_filename)
print("Datasets ready:", len(train_dataset), "train rows,", len(val_dataset), "val rows")

## 6. Unified encoder/decoder framework

Same building blocks as the architecture-search notebook, but the decoders now expose the knobs we tune. `GPT2Decoder` takes a `dropout` and a `freeze_base` flag (freeze the pretrained GPT-2 weights and train only the cross-attention + projection). `build_model(config)` constructs any combination from an `ExperimentConfig`.

In [ ]:
# ---- Word-level vocabulary (for the GRU decoder) ----------------------------
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def numericalize(self, cap):
        ids = [self.word2idx["<start>"]]
        ids += [self.word2idx.get(t, self.word2idx["<unk>"]) for t in word_tokenize(cap.lower())]
        ids.append(self.word2idx["<end>"])
        return ids

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = Vocabulary(freq_threshold=GRU_FREQ_THRESHOLD)
rnn_vocab.build([a["caption"] for a in train_annotations])
print("GRU word-level vocab size:", len(rnn_vocab))

# ---- Image preprocessing ----------------------------------------------------
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) ------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # (B,2048,7,7)
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)   # (B, 49, 2048)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state                       # (B, T, 768)

# ---- Decoder A: word-level GRU ----------------------------------------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))
        self.criterion = nn.CrossEntropyLoss(ignore_index=self.pad_id)

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)
        return self.bn(self.img_proj(pooled))

    def forward(self, enc_seq, captions):
        feat = self._img_token(enc_seq)
        emb = self.dropout(self.embed(captions[:, :-1]))
        inputs = torch.cat((feat.unsqueeze(1), emb), dim=1)
        hiddens, _ = self.rnn(inputs)
        logits = self.linear(hiddens)
        ml = min(logits.size(1), captions.size(1))
        return self.criterion(
            logits[:, :ml, :].reshape(-1, logits.size(-1)),
            captions[:, :ml].reshape(-1))

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        feat = self._img_token(enc_seq)
        B = feat.size(0)
        inp = feat.unsqueeze(1)
        states = None
        done = torch.zeros(B, dtype=torch.bool, device=feat.device)
        seqs = [[] for _ in range(B)]
        for _ in range(max_len):
            hiddens, states = self.rnn(inp, states)
            pred = self.linear(hiddens.squeeze(1)).argmax(1)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(pred[i].item())
                if tid == self.end_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            inp = self.embed(pred).unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention (tunable dropout / freeze) --------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))   # dedicated pad token
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id
        if freeze_base:
            # Train only the (random-init) cross-attention + enc_proj; freeze the
            # rest of the pretrained GPT-2.
            for n, p in self.gpt2.named_parameters():
                p.requires_grad = ("crossattention" in n) or ("ln_cross_attn" in n)

    def forward(self, enc_seq, token_ids):
        enc_hidden = self.enc_proj(enc_seq)
        attn = (token_ids != self.pad_id).long()
        labels = token_ids.clone()
        labels[token_ids == self.pad_id] = -100
        out = self.gpt2(input_ids=token_ids, attention_mask=attn,
                        encoder_hidden_states=enc_hidden, labels=labels)
        return out.loss

    @torch.no_grad()
    def generate(self, enc_seq, max_len):
        enc_hidden = self.enc_proj(enc_seq)
        B = enc_hidden.size(0)
        dev = enc_hidden.device
        cur = torch.full((B, 1), self.bos_id, dtype=torch.long, device=dev)
        seqs = [[] for _ in range(B)]
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        past = None
        for _ in range(max_len):
            out = self.gpt2(input_ids=cur, encoder_hidden_states=enc_hidden,
                            past_key_values=past, use_cache=True)
            past = out.past_key_values
            nxt = out.logits[:, -1, :].argmax(-1)
            for i in range(B):
                if done[i]:
                    continue
                tid = int(nxt[i].item())
                if tid == self.eos_id:
                    done[i] = True
                else:
                    seqs[i].append(tid)
            if bool(done.all()):
                break
            cur = nxt.unsqueeze(1)
        return seqs

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- Full model -------------------------------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    def _encode(self, images):
        if self.freeze_encoder:
            with torch.no_grad():
                return self.encoder(images)
        return self.encoder(images)

    def forward(self, images, labels):
        return self.decoder(self._encode(images), labels)

    @torch.no_grad()
    def generate(self, images, max_len):
        return self.decoder.generate(self._encode(images), max_len)

    def decode(self, ids):
        return self.decoder.decode(ids)

## 7. Config, builder, training, and BLEU evaluation

An `ExperimentConfig` fully describes a run (architecture + hyperparameters + budget). `run_experiment` runs the shared training loop with **per-epoch best-BLEU checkpointing, early stopping, and rollback** — identical in spirit to the GRU notebook — and returns the per-epoch history so we can plot all models together.

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    encoder_kind: str            # "cnn" | "vit" | "clip"
    encoder_name: str
    decoder: str                 # "gpt2" | "gru"
    learning_rate: float
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    # GRU-only dims
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    freq_threshold: int = 5
    # budget
    batch_size: int = 16
    epochs: int = 15
    max_train_batches: Optional[int] = 300   # None = use the full train set each epoch


def build_model(config):
    encoder = ImageEncoder(config.encoder_kind, config.encoder_name)
    feat_dim = encoder.feat_dim
    if config.decoder == "gru":
        decoder = GRUDecoder(feat_dim, rnn_vocab, config.embed_size,
                             config.hidden_size, config.num_layers, config.dropout)
    elif config.decoder == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
        decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                              freeze_base=config.freeze_gpt2_base)
    else:
        raise ValueError(f"Unknown decoder: {config.decoder}")

    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    for p in model.encoder.parameters():
        p.requires_grad = False
    image_processor = (None if config.encoder_kind == "cnn"
                       else AutoImageProcessor.from_pretrained(config.encoder_name))
    return {"model": model, "encoder_kind": config.encoder_kind,
            "decoder_kind": config.decoder, "image_processor": image_processor}


def make_collate(bundle):
    kind = bundle["encoder_kind"]
    dk = bundle["decoder_kind"]
    image_processor = bundle["image_processor"]
    tok = bundle["model"].decoder.tokenizer if dk == "gpt2" else None

    def collate(batch):
        images, captions, image_ids = zip(*batch)
        pixel_values = preprocess_images(images, kind, image_processor)
        if dk == "gru":
            seqs = [rnn_vocab.numericalize(c) for c in captions]
            maxlen = max(len(s) for s in seqs)
            labels = torch.zeros(len(seqs), maxlen, dtype=torch.long)
            for i, s in enumerate(seqs):
                labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        else:
            enc = tok(list(captions), add_special_tokens=False,
                      truncation=True, max_length=MAX_TEXT_LEN - 2)
            seqs = [[tok.bos_token_id] + ids + [tok.eos_token_id] for ids in enc["input_ids"]]
            maxlen = max(len(s) for s in seqs)
            labels = torch.full((len(seqs), maxlen), tok.pad_token_id, dtype=torch.long)
            for i, s in enumerate(seqs):
                labels[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        return pixel_values, labels, torch.tensor(image_ids, dtype=torch.long)
    return collate


def train_one_epoch(model, loader, optimizer, device, max_batches=None):
    model.train()
    total_loss, n = 0.0, 0
    for batch_idx, (pixel_values, labels, _) in enumerate(loader):
        if max_batches is not None and batch_idx >= max_batches:
            break
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        loss = model(pixel_values, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        total_loss += float(loss.item()); n += 1
    return total_loss / max(n, 1)


def _tok(text):
    return word_tokenize(text.lower())


@torch.no_grad()
def evaluate_bleu(model, loader, device, num_batches=None):
    model.eval()
    references, hypotheses, seen = [], [], set()
    smoothing = SmoothingFunction().method1
    for batch_idx, (pixel_values, labels, image_ids) in enumerate(loader):
        if num_batches is not None and batch_idx >= num_batches:
            break
        pixel_values = pixel_values.to(device)
        gen_ids = model.generate(pixel_values, MAX_GEN_LEN)
        for j in range(pixel_values.size(0)):
            img_id = int(image_ids[j].item())
            if img_id in seen:
                continue
            seen.add(img_id)
            hypotheses.append(_tok(model.decode(gen_ids[j])))
            references.append([_tok(c) for c in img_id_to_captions[img_id]])
    return 0.0 if not hypotheses else corpus_bleu(references, hypotheses, smoothing_function=smoothing)


def run_experiment(config, save_ckpt=True, verbose=True):
    """Shared loop: per-epoch best-BLEU checkpoint, early stopping, rollback.
    Returns (result_dict, bundle, history)."""
    if verbose:
        print("\n" + "=" * 80); print("Running:", config.name); print("=" * 80)

    bundle = build_model(config)
    model = bundle["model"].to(device)
    collate = make_collate(bundle)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True,
                              num_workers=NUM_WORKERS, collate_fn=collate,
                              pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False,
                            num_workers=NUM_WORKERS, collate_fn=collate,
                            pin_memory=torch.cuda.is_available())

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=config.learning_rate,
                                  weight_decay=config.weight_decay)

    ckpt_path = os.path.join(OUTPUT_DIR, f"{config.name}.pt")
    losses, val_bleus = [], []
    best_bleu, best_epoch, best_state = -1.0, -1, None
    epochs_no_improve = 0
    start = time.time()

    for epoch in range(config.epochs):
        loss = train_one_epoch(model, train_loader, optimizer, device,
                               max_batches=config.max_train_batches)
        losses.append(loss)
        val_bleu = evaluate_bleu(model, val_loader, device, num_batches=EVAL_NUM_BATCHES)
        val_bleus.append(val_bleu)

        improved = val_bleu > best_bleu + EARLY_STOP_MIN_DELTA
        flag = ""
        if improved:
            best_bleu, best_epoch = val_bleu, epoch
            epochs_no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
            if save_ckpt:
                torch.save({"state_dict": best_state, "config": asdict(config),
                            "bleu4": best_bleu, "epoch": epoch + 1}, ckpt_path)
            flag = "  <-- best so far" + (" (saved)" if save_ckpt else "")
        else:
            epochs_no_improve += 1

        if verbose:
            print(f"Epoch {epoch+1}/{config.epochs}: train loss = {loss:.4f}, "
                  f"val BLEU-4 = {val_bleu:.4f}{flag}")

        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            if verbose:
                print(f"Early stopping: no improvement for {EARLY_STOP_PATIENCE} "
                      f"epoch(s). Best was epoch {best_epoch+1} (BLEU-4 = {best_bleu:.4f}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    result = {
        "name": config.name,
        "encoder": f"{config.encoder_kind} ({config.encoder_name})",
        "decoder": config.decoder,
        "learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay,
        "dropout": config.dropout,
        "freeze_gpt2_base": config.freeze_gpt2_base,
        "best_epoch": best_epoch + 1,
        "epochs_trained": len(losses),
        "final_train_loss": losses[-1] if losses else None,
        "bleu4": best_bleu,
        "train_time_seconds": round(time.time() - start, 2),
        "trainable_params": sum(p.numel() for p in params),
        "checkpoint": ckpt_path if save_ckpt else "",
    }
    history = {"losses": losses, "val_bleus": val_bleus}
    return result, bundle, history

## 8. Phase 1 — hyperparameter search (CNN+GPT-2, ViT+GPT-2, CLIP+GPT-2)

A randomized grid search over the GPT-2 decoder's knobs, mirroring the GRU notebook's search: sample `MAX_TRIALS` configs from the grid and train each briefly (`GRID_EPOCHS` x `GRID_MAX_TRAIN_BATCHES`). The best config per architecture is carried into the full 15-epoch training.

In [ ]:
ARCH_TARGETS = [
    {"label": "cnn_gpt2", "encoder_kind": "cnn",
     "encoder_name": "resnet50"},
    {"label": "vit_gpt2", "encoder_kind": "vit",
     "encoder_name": "google/vit-base-patch16-224-in21k"},
    {"label": "clip_gpt2", "encoder_kind": "clip",
     "encoder_name": "openai/clip-vit-base-patch32"},
]

# GPT-2 decoder hyperparameter grid (full grid = 3*2*2*2 = 24 combinations).
GPT2_PARAM_GRID = {
    "learning_rate":     [5e-5, 1e-4, 3e-5],
    "weight_decay":      [0.0, 0.01],
    "dropout":           [0.1, 0.2],
    "freeze_gpt2_base":  [False, True],   # True = train only cross-attn + enc_proj
}

def sample_gpt2_configs(arch, max_trials, epochs, max_batches):
    keys = list(GPT2_PARAM_GRID.keys())
    combos = list(itertools.product(*GPT2_PARAM_GRID.values()))
    _random.Random(SEED).shuffle(combos)
    chosen = combos if max_trials is None else combos[:max_trials]
    cfgs = []
    for i, combo in enumerate(chosen):
        p = dict(zip(keys, combo))
        name = (f"{arch['label']}__t{i:02d}_lr{p['learning_rate']:g}_wd{p['weight_decay']}"
                f"_d{p['dropout']}_{'froz' if p['freeze_gpt2_base'] else 'full'}")
        cfgs.append(ExperimentConfig(
            name=name, encoder_kind=arch["encoder_kind"], encoder_name=arch["encoder_name"],
            decoder="gpt2", learning_rate=p["learning_rate"], weight_decay=p["weight_decay"],
            dropout=p["dropout"], freeze_gpt2_base=p["freeze_gpt2_base"],
            batch_size=BATCH_SIZE, epochs=epochs, max_train_batches=max_batches))
    return cfgs

_full = len(list(itertools.product(*GPT2_PARAM_GRID.values())))
print(f"Full GPT-2 grid: {_full} combinations | sampling {MAX_TRIALS} per architecture")
print("Architectures to search:", ", ".join(a["label"] for a in ARCH_TARGETS))

In [ ]:
search_results = []
best_per_arch = {}

for arch in ARCH_TARGETS:
    cfgs = sample_gpt2_configs(arch, MAX_TRIALS, GRID_EPOCHS, GRID_MAX_TRAIN_BATCHES)
    print("\n" + "#" * 80)
    print(f"# Hyperparameter search: {arch['label']}  ({len(cfgs)} trials)")
    print("#" * 80)
    arch_best = None
    for cfg in cfgs:
        result, bundle, _ = run_experiment(cfg, save_ckpt=False)
        search_results.append(result)
        del bundle["model"]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if arch_best is None or result["bleu4"] > arch_best["bleu4"]:
            arch_best = result
    best_per_arch[arch["label"]] = arch_best
    print(f"\n>> Best {arch['label']}: lr={arch_best['learning_rate']:g}, "
          f"wd={arch_best['weight_decay']}, dropout={arch_best['dropout']}, "
          f"freeze_base={arch_best['freeze_gpt2_base']}  ->  BLEU-4 {arch_best['bleu4']:.4f}")

search_df = pd.DataFrame(search_results).sort_values("bleu4", ascending=False).reset_index(drop=True)
search_csv = os.path.join(OUTPUT_DIR, "hyperparam_search.csv")
search_df.to_csv(search_csv, index=False)
print("\nSaved search results:", search_csv)
search_df

## 9. Phase 2 — full training (15 epochs + early stopping)

The winning GPT-2 config for each architecture is retrained for `EPOCHS` epochs with early stopping and per-epoch best-weight checkpointing. The earlier **GRU** winner is retrained with the *same* recipe so all three curves are directly comparable. Per-epoch histories are kept for the combined plot.

In [ ]:
final_configs = []

# Transformer winners (best config from the search), retrained fully.
for arch in ARCH_TARGETS:
    b = best_per_arch[arch["label"]]
    final_configs.append(ExperimentConfig(
        name=f"{arch['label']}_best",
        encoder_kind=arch["encoder_kind"], encoder_name=arch["encoder_name"],
        decoder="gpt2", learning_rate=b["learning_rate"], weight_decay=b["weight_decay"],
        dropout=b["dropout"], freeze_gpt2_base=b["freeze_gpt2_base"],
        batch_size=BATCH_SIZE, epochs=EPOCHS, max_train_batches=FULL_TRAIN_MAX_BATCHES))

# Earlier GRU winner config, retrained with the SAME 15-epoch + early-stop recipe.
final_configs.append(ExperimentConfig(
    name="cnn_gru_baseline",
    encoder_kind="cnn", encoder_name="resnet50", decoder="gru",
    learning_rate=GRU_LR, dropout=GRU_DROPOUT,
    embed_size=GRU_EMBED_SIZE, hidden_size=GRU_HIDDEN_SIZE,
    num_layers=GRU_NUM_LAYERS, freq_threshold=GRU_FREQ_THRESHOLD,
    batch_size=BATCH_SIZE, epochs=EPOCHS, max_train_batches=FULL_TRAIN_MAX_BATCHES))

print("Final training plan:")
for c in final_configs:
    extra = (f"lr={c.learning_rate:g}, wd={c.weight_decay}, dropout={c.dropout}, "
             f"freeze_base={c.freeze_gpt2_base}" if c.decoder == "gpt2"
             else f"lr={c.learning_rate:g}, h{c.hidden_size}/e{c.embed_size}/L{c.num_layers}/d{c.dropout}")
    print(f"  - {c.name}: {c.decoder} | {extra}")

final_results, histories = [], {}
best_bundle, best_overall = None, -1

for cfg in final_configs:
    result, bundle, hist = run_experiment(cfg, save_ckpt=True)
    final_results.append(result)
    histories[cfg.name] = hist
    if result["bleu4"] > best_overall:
        best_overall = result["bleu4"]
        best_bundle = {**bundle, "name": cfg.name}
    else:
        del bundle["model"]
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

final_df = pd.DataFrame(final_results).sort_values("bleu4", ascending=False).reset_index(drop=True)
final_csv = os.path.join(OUTPUT_DIR, "final_training_results.csv")
final_df.to_csv(final_csv, index=False)
print("\nSaved final results:", final_csv)
print("Best overall:", best_bundle["name"], "| BLEU-4 =", round(best_overall, 4))
final_df

## 10. Comparison plot — all models on one graph

Validation BLEU-4 per epoch for **CNN+GPT-2**, **ViT+GPT-2**, **CLIP+GPT-2**, and the **GRU** baseline, drawn on the same axes. (A train-loss plot is included too, but note GRU vs GPT-2 losses use different vocabularies/criteria, so only the BLEU-4 curves are strictly comparable.)

In [ ]:
DISPLAY = {"cnn_gpt2_best": "CNN + GPT-2",
           "vit_gpt2_best": "ViT + GPT-2",
           "clip_gpt2_best": "CLIP + GPT-2",
           "cnn_gru_baseline": "CNN + GRU (baseline)"}

# ---- Validation BLEU-4 (the head-to-head comparison) ------------------------
plt.figure(figsize=(9, 6))
for name, hist in histories.items():
    epochs_axis = range(1, len(hist["val_bleus"]) + 1)
    plt.plot(epochs_axis, hist["val_bleus"], marker="o", linewidth=2,
             label=DISPLAY.get(name, name))
plt.title("Validation BLEU-4 over training (15 epochs, early stopping)")
plt.xlabel("Epoch"); plt.ylabel("Validation BLEU-4")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
bleu_plot = os.path.join(OUTPUT_DIR, "bleu4_comparison.png")
plt.savefig(bleu_plot, dpi=150); plt.show()
print("Saved:", bleu_plot)

# ---- Training loss (per-decoder scale; shown for completeness) --------------
plt.figure(figsize=(9, 6))
for name, hist in histories.items():
    epochs_axis = range(1, len(hist["losses"]) + 1)
    plt.plot(epochs_axis, hist["losses"], marker="s", linewidth=2,
             label=DISPLAY.get(name, name))
plt.title("Training loss over training")
plt.xlabel("Epoch"); plt.ylabel("Train loss")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
loss_plot = os.path.join(OUTPUT_DIR, "loss_comparison.png")
plt.savefig(loss_plot, dpi=150); plt.show()
print("Saved:", loss_plot)

## 11. Save the best model to Google Drive (persistent storage)

`OUTPUT_DIR` is on the ephemeral Colab VM disk. This copies the best model's checkpoint, the result CSVs, and the comparison plots to Drive. On a local Jupyter it just reports the local path.

In [ ]:
import shutil

DRIVE_SAVE_DIR = "/content/drive/MyDrive/image_captioning_finetune"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    best_ckpt = os.path.join(OUTPUT_DIR, f"{best_bundle['name']}.pt")
    for fpath in [best_ckpt, search_csv, final_csv, bleu_plot, loss_plot]:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(DRIVE_SAVE_DIR, os.path.basename(fpath)))
    print("Saved best checkpoint + CSVs + plots to Drive:", DRIVE_SAVE_DIR)
except ImportError:
    print("Not on Colab — everything already persists locally under:", os.path.abspath(OUTPUT_DIR))

## 12. Report-ready notes

- **What this run does:** for the strongest transformer combos from the architecture search (**CNN+GPT-2**, **ViT+GPT-2**, **CLIP+GPT-2**) it runs a short randomized hyperparameter search over the GPT-2 decoder (learning rate, weight decay, dropout, freeze-base vs full fine-tune), then retrains each winner for **15 epochs with early stopping** (monitor val BLEU-4, patience 3) keeping the per-epoch best weights. The earlier **GRU** winner is retrained with the same recipe.
- **Fair comparison:** identical `BATCH_SIZE`, `FULL_TRAIN_MAX_BATCHES`, eval procedure, and frozen encoders across all runs; only the encoder/decoder and (for GPT-2) the searched hyperparameters differ.
- **Knobs to scale up:** raise `MAX_TRIALS` for a wider search, set `FULL_TRAIN_MAX_BATCHES = None` to train on the full subset each epoch, or switch `COCO_SPLIT` to `train` for the full dataset.
- **Outputs:** `hyperparam_search.csv`, `final_training_results.csv`, `bleu4_comparison.png`, `loss_comparison.png`, and one `*.pt` checkpoint per final model (best epoch).